# Six 4-fish recordings: four assignment pipelines on all EODs, and refractory violations

Recordings (`model_voltages_4fish.RECS`): three tracked by Shiv (ZEUS / `.mgz` masks): 2024-02-20 21_16_26 (the recording of
`eod_two_layer_pipeline_clean.ipynb`), 2024-02-20 18_16_24 and 2024-10-23 15_18_28; three tracked by Kaden: 2024-02-20 12_32_58,
15_16_23 and 17_56_24. All EODs of each (the `_no_ref_per` tables: detected without a refractory period).

Pipelines:
1. **numerical, shape**: my model (BEM conductivities: interior 100 × water, skin 0.05 × water over 0.005 cm; sink at the tail tip;
   unit current, no length scaling), sign fixed (× −1); model and recording each scaled to max |v| = 1 over the 10 pairs; smallest L2
   wins.
2. **numerical, voltage, e = 0**: $\lVert r - g\,m_f\rVert$ in volts, the same $g$ for every fish: the emitter's median gain in the
   control of the same period (2-fish, 2024-02-13, for February; 3-fish, 2024-10-15, for October).
3. **numerical, voltage, $g_0 L^{0.25}$**: $\lVert r - g_0 L_f^{0.25} m_f\rVert$, $g_0$ of the control of the same period
   (`EOD_baseline_g0.ipynb`), L in cm (`fish_lengths.zeus_lengths` rules; Kaden's recordings: `results/fish_lengths_<time>.json`).
4. **analytic (Denis)**: his `assign_eods` (`EODAssignment/analysis/social_electric_fish_runners/denis_assign_4fish.py`): his keypoint midline and field model, the recording's
   electrode file, no refractory constraint, fish = the CSV's tracking letters.

Model voltages: `model_voltages_4fish.py` (one solve per fish per EOD frame, mask frame = Frame). A fish without a mask or midline
cannot win; an EOD with no fish to compare is unassigned.

**Refractory violation**: an EOD less than 10 ms after the previous detected EOD (any fish), with both EODs assigned to the same
fish. Reported as % of all EODs, and as % of the EODs that follow the previous one within 10 ms. **Chance**: the same % if the labels
were shuffled with each fish's share kept (Σ p_f² × the % of EODs within 10 ms).

In [1]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import linear_sum_assignment
%matplotlib inline
from model_voltages_4fish import RECS, OUT, N
from eod_assign import peak_pattern

REFRACTORY_S = 0.010
PX_PER_CM = 912 / 60
E = 0.25
G0 = pd.read_csv('results/baseline_g0/g0_controls.csv').set_index(['recording', 'EODs']).xs('all better-list EODs', level='EODs')
CONTROL = {'Feb': '2-fish (2024-02-13)', 'Oct': '3-fish (2024-10-15)'}
GAIN = {p: dict(g_e0=G0.loc[c, 'median g'], g0=G0.loc[c, 'g0 = median g / L^e']) for p, c in CONTROL.items()}
display(pd.DataFrame(GAIN).T.rename_axis('period').round(4))

PIPES = {   # label: (colour, marker)
    'numerical, shape': ('#C8361E', 'o'),
    'numerical, voltage, e = 0': ('#F08A24', 'o'),
    'numerical, voltage, g0 L^0.25': ('#E8C547', 'o'),
    'analytic (Denis)': ('#1F5FBF', 'D'),
}

def lengths_cm(rec):
    if RECS[rec]['identity'] == 'kaden':
        j = json.load(open(RECS[rec]['lengths']))['fish']; return np.array([j[l]['L_px'] for l in 'ABCD']) / PX_PER_CM
    j = json.load(open(f'results/eod_gain_4fish/{rec}/fish_lengths_4fish.json'))['fish']
    return np.array([j[str(f)]['L_px'] for f in range(4)]) / PX_PER_CM

L = {rec: lengths_cm(rec) for rec in RECS}
display(pd.DataFrame(L, index=[f'fish {i}' for i in range(4)]).T.round(2).assign(tracking=[RECS[r]['tracking'] for r in RECS],
                                                                                   period=[RECS[r]['period'] for r in RECS]))

,g_e0,g0
period,,
Feb,0.4925,0.2474
Oct,0.9152,0.4938


,fish 0,fish 1,fish 2,fish 3,tracking,period
2024-02-20T21_16_26,14.74,12.95,9.36,8.18,Shiv,Feb
2024-02-20T18_16_24,14.40,12.85,9.17,8.26,Shiv,Feb
2024-10-23T15_18_28,18.08,12.90,12.90,8.62,Shiv,Oct
2024-02-20T12_32_58,15.54,13.02,8.73,8.29,Kaden,Feb
2024-02-20T15_16_23,15.58,12.72,8.77,8.44,Kaden,Feb
2024-02-20T17_56_24,15.29,12.73,9.46,8.12,Kaden,Feb


## Assignments

In [2]:
def pick(dist):
    dist = np.where(np.isfinite(dist), dist, np.inf)
    return np.where(np.isfinite(dist.min(1)), dist.argmin(1), -1)

A, S = {}, []
for rec, cfg in RECS.items():
    df = pd.read_csv(cfg['eod'], usecols=lambda c: c in ('Frame', 'RelativeTime') or c.startswith('electrode_'))
    pairs = sorted({int(c.split('_')[1]) for c in df.columns if c.startswith('electrode_')})
    V = np.stack([df[[f'electrode_{p}_time_{t}' for t in range(1, 26)]].to_numpy(float) for p in pairs], axis=1)   # (n, 10, 25)
    V -= np.nanmean(V, axis=2, keepdims=True)
    t_star = np.nanargmax(np.abs(V).reshape(len(V), -1), axis=1) % V.shape[2]                     # = eod_assign.peak_pattern
    R = V[np.arange(len(V)), :, t_star]
    assert np.allclose(R[0], peak_pattern(V[0])[0])
    cache = pd.read_pickle(f'{OUT}/{rec}/model_voltages_N{N}_sink0_bem_by_frame.pkl')
    M = -np.stack([cache[int(k)] for k in df.Frame])                                              # (n, 4, 10), sign fixed
    unit = lambda x: x / np.abs(x).max(axis=-1, keepdims=True)
    g = GAIN[cfg['period']]
    a = pd.DataFrame({'Frame': df.Frame, 'time': df.RelativeTime})
    with np.errstate(invalid='ignore', divide='ignore'):
        a['numerical, shape'] = pick(np.linalg.norm(unit(M) - unit(R)[:, None], axis=2))
    a['numerical, voltage, e = 0'] = pick(np.linalg.norm(R[:, None] - g['g_e0'] * M, axis=2))
    a['numerical, voltage, g0 L^0.25'] = pick(np.linalg.norm(R[:, None] - g['g0'] * L[rec][None, :, None] ** E * M, axis=2))
    d = pd.read_pickle(f'{OUT}/denis/Assigned_Data_4fish_{rec}.pkl')
    assert len(d) == len(df) and (d.Frame.to_numpy() == df.Frame.to_numpy()).all()
    a['analytic (Denis)'] = d.discharging_fish.map({l: i for i, l in enumerate('ABCD')}).fillna(-1).astype(int).to_numpy()
    a = a.sort_values('time', kind='stable')
    a['dt_prev_s'] = a.time.diff()
    a.to_csv(f'{OUT}/{rec}/eod_assignments_4pipelines.csv', index_label='row')
    A[rec] = a
    S.append({'recording': rec, 'tracking': cfg['tracking'], 'EODs': len(a), 'EOD frames': a.Frame.nunique(),
              'min(model fish per EOD)': int(np.isfinite(M).all(2).sum(1).min()), 'EODs with all 4 models': int(np.isfinite(M).all((1, 2)).sum())})
display(pd.DataFrame(S).set_index('recording'))

,tracking,EODs,EOD frames,min(model fish per EOD),EODs with all 4 models
recording,,,,,
2024-02-20T21_16_26,Shiv,54562,38930,3,54358
2024-02-20T18_16_24,Shiv,73355,46413,2,70969
2024-10-23T15_18_28,Shiv,128553,57181,2,127473
2024-02-20T12_32_58,Kaden,21810,16518,3,20815
2024-02-20T15_16_23,Kaden,59496,41341,3,59347
2024-02-20T17_56_24,Kaden,70971,45711,2,68101


### Share of EODs per fish

The fish numbers of the numerical pipelines (masks: size rank or ZEUS / Kaden labels) and of Denis' (tracking letters) are not the
same fish in Shiv's recordings, so the shares are listed per pipeline as they are.

In [3]:
share = pd.concat({rec: pd.DataFrame({p: a[p].value_counts(normalize=True).reindex([-1, 0, 1, 2, 3], fill_value=0) * 100 for p in PIPES}).T
                   for rec, a in A.items()}, names=['recording', 'pipeline'])
share.columns = ['unassigned', 'fish 0 / A', 'fish 1 / B', 'fish 2 / C', 'fish 3 / D']
share.to_csv(f'{OUT}/share_per_fish_4pipelines.csv'); display(share.round(1))

unassigned  fish 0 / A  \
recording           pipeline                                                
2024-02-20T21_16_26 numerical, shape                      0.0        32.3   
                    numerical, voltage, e = 0             0.0        30.3   
                    numerical, voltage, g0 L^0.25         0.0        30.1   
                    analytic (Denis)                      0.0        32.3   
2024-02-20T18_16_24 numerical, shape                      0.0        29.8   
                    numerical, voltage, e = 0             0.0        26.8   
                    numerical, voltage, g0 L^0.25         0.0        26.8   
                    analytic (Denis)                      0.0        29.4   
2024-10-23T15_18_28 numerical, shape                      0.0        33.5   
                    numerical, voltage, e = 0             0.0        28.8   
                    numerical, voltage, g0 L^0.25         0.0        27.4   
                    analytic (Denis)                      0.0        31.3   
2024-02-20T12_32_58 numerical, shape                      0.0        27.0   
                    numerical, voltage, e = 0             0.0        25.3   
                    numerical, voltage, g0 L^0.25         0.0        25.3   
                    analytic (Denis)                      0.0        27.0   
2024-02-20T15_16_23 numerical, shape                      0.0        28.3   
                    numerical, voltage, e = 0             0.0        26.6   
                    numerical, voltage, g0 L^0.25         0.0        26.4   
                    analytic (Denis)                      0.0        28.1   
2024-02-20T17_56_24 numerical, shape                      0.0        31.8   
                    numerical, voltage, e = 0             0.0        29.1   
                    numerical, voltage, g0 L^0.25         0.0        28.9   
                    analytic (Denis)                      0.0        31.4   

                                                   fish 1 / B  fish 2 / C  \
recording           pipeline                                                
2024-02-20T21_16_26 numerical, shape                     16.6        26.1   
                    numerical, voltage, e = 0            12.4        28.9   
                    numerical, voltage, g0 L^0.25        13.0        28.4   
                    analytic (Denis)                     16.1        26.2   
2024-02-20T18_16_24 numerical, shape                     18.0        26.1   
                    numerical, voltage, e = 0            14.4        26.6   
                    numerical, voltage, g0 L^0.25        14.6        26.6   
                    analytic (Denis)                     17.6        26.2   
2024-10-23T15_18_28 numerical, shape                     24.1        21.9   
                    numerical, voltage, e = 0            21.6        21.9   
                    numerical, voltage, g0 L^0.25        21.5        22.1   
                    analytic (Denis)                     23.7        22.6   
2024-02-20T12_32_58 numerical, shape                     25.2        26.0   
                    numerical, voltage, e = 0            23.0        27.2   
                    numerical, voltage, g0 L^0.25        23.2        27.3   
                    analytic (Denis)                     24.1        26.3   
2024-02-20T15_16_23 numerical, shape                     18.4        24.2   
                    numerical, voltage, e = 0            14.9        25.2   
                    numerical, voltage, g0 L^0.25        15.3        25.1   
                    analytic (Denis)                     17.6        24.9   
2024-02-20T17_56_24 numerical, shape                     12.3        28.5   
                    numerical, voltage, e = 0            10.2        29.9   
                    numerical, voltage, g0 L^0.25        10.1        29.8   
                    analytic (Denis)                     11.7        28.1   

                                             

### Agreement between pipelines

% of EODs on which two pipelines give the same fish. Denis' letters are matched to the numerical fish numbers first, by the
pairing with the most EODs in common (Hungarian method on the co-assignment counts, per recording).

In [4]:
def matched(x, y):
    '''y relabelled to x's labels by the pairing with the most common EODs.'''
    C = np.array([[((x == i) & (y == j)).sum() for j in range(4)] for i in range(4)])
    r, c = linear_sum_assignment(-C); m = {int(j): int(i) for i, j in zip(r, c)}
    return np.array([m.get(int(v), -1) for v in y])

agree = {}
for rec, a in A.items():
    den = matched(a['numerical, shape'].to_numpy(), a['analytic (Denis)'].to_numpy())
    cols = {p: a[p].to_numpy() for p in PIPES if p != 'analytic (Denis)'} | {'analytic (Denis)': den}
    agree[rec] = {f'{p} vs {q}': 100 * (cols[p] == cols[q]).mean() for i, p in enumerate(PIPES) for q in list(PIPES)[i + 1:]}
AG = pd.DataFrame(agree).T; AG.to_csv(f'{OUT}/agreement_4pipelines.csv'); display(AG.round(1))

,"numerical, shape vs numerical, voltage, e = 0","numerical, shape vs numerical, voltage, g0 L^0.25","numerical, shape vs analytic (Denis)","numerical, voltage, e = 0 vs numerical, voltage, g0 L^0.25","numerical, voltage, e = 0 vs analytic (Denis)","numerical, voltage, g0 L^0.25 vs analytic (Denis)"
2024-02-20T21_16_26,89.7,90.9,91.1,98.2,85.9,86.9
2024-02-20T18_16_24,87.8,88.6,89.6,98.4,84.9,85.5
2024-10-23T15_18_28,86.5,85.2,82.5,97.8,78.0,77.1
2024-02-20T12_32_58,95.1,95.4,96.9,99.1,94.2,94.6
2024-02-20T15_16_23,89.7,90.3,90.8,98.4,87.7,88.2
2024-02-20T17_56_24,88.2,89.1,91.3,98.2,86.3,87.0


## Refractory violations

In [5]:
rows = []
for rec, a in A.items():
    close = (a.dt_prev_s < REFRACTORY_S).to_numpy()
    for p in PIPES:
        f = a[p].to_numpy(); prev = np.r_[-2, f[:-1]]
        viol = close & (f >= 0) & (f == prev)
        pf = np.bincount(f[f >= 0], minlength=4) / max((f >= 0).sum(), 1)
        rows.append({'recording': rec, 'tracking': RECS[rec]['tracking'], 'pipeline': p, 'EODs': len(a),
                     '% EODs within 10 ms of the previous': 100 * close.mean(),
                     '% violations (of all EODs)': 100 * viol.mean(),
                     '% violations (of EODs within 10 ms)': 100 * viol.sum() / close.sum(),
                     'chance, % of all EODs': 100 * close.mean() * (pf ** 2).sum(),
                     'chance, % of EODs within 10 ms': 100 * (pf ** 2).sum()})
RV = pd.DataFrame(rows); RV.to_csv(f'{OUT}/refractory_violations_4pipelines.csv', index=False)
display(RV.pivot_table(index='recording', columns='pipeline', values='% violations (of all EODs)', sort=False).round(2))
display(RV.pivot_table(index='recording', columns='pipeline', values='% violations (of EODs within 10 ms)', sort=False).round(1))
display(RV.pivot_table(index='recording', columns='pipeline', values='chance, % of EODs within 10 ms', sort=False).round(1))
tot = RV.assign(v=RV['% violations (of all EODs)'] * RV.EODs / 100).groupby('pipeline', sort=False).apply(
    lambda t: pd.Series({'% violations, all six recordings': 100 * t.v.sum() / t.EODs.sum()}), include_groups=False)
display(tot.round(2))

pipeline,"numerical, shape","numerical, voltage, e = 0","numerical, voltage, g0 L^0.25",analytic (Denis)
recording,,,,
2024-02-20T21_16_26,1.81,2.28,1.98,1.73
2024-02-20T18_16_24,3.49,3.47,3.25,3.43
2024-10-23T15_18_28,7.44,5.34,5.56,8.20
2024-02-20T12_32_58,0.71,1.15,1.10,0.32
2024-02-20T15_16_23,3.12,3.09,3.04,2.70
2024-02-20T17_56_24,3.36,3.78,3.48,2.90


pipeline,"numerical, shape","numerical, voltage, e = 0","numerical, voltage, g0 L^0.25",analytic (Denis)
recording,,,,
2024-02-20T21_16_26,6.4,8.1,7.0,6.1
2024-02-20T18_16_24,9.3,9.3,8.7,9.2
2024-10-23T15_18_28,12.2,8.7,9.1,13.4
2024-02-20T12_32_58,2.9,4.7,4.4,1.3
2024-02-20T15_16_23,10.1,10.1,9.9,8.8
2024-02-20T17_56_24,9.3,10.4,9.6,8.0


pipeline,"numerical, shape","numerical, voltage, e = 0","numerical, voltage, g0 L^0.25",analytic (Denis)
recording,,,,
2024-02-20T21_16_26,26.3,27.1,27.0,26.3
2024-02-20T18_16_24,25.7,26.7,26.6,25.8
2024-10-23T15_18_28,26.0,25.4,25.4,25.5
2024-02-20T12_32_58,25.2,25.1,25.1,25.1
2024-02-20T15_16_23,25.7,26.7,26.6,25.8
2024-02-20T17_56_24,27.2,27.9,28.0,27.4


,"% violations, all six recordings"
pipeline,
"numerical, shape",4.28
"numerical, voltage, e = 0",3.77
"numerical, voltage, g0 L^0.25",3.70
analytic (Denis),4.34


In [6]:
plt.rcParams.update({'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True, 'grid.color': '#e6e6e3'})
recs = list(RECS); x = np.arange(len(recs)); w = 0.2
fig, axs = plt.subplots(2, 1, figsize=(14, 10))
for ax, col, ch in [(axs[0], '% violations (of all EODs)', 'chance, % of all EODs'),
                    (axs[1], '% violations (of EODs within 10 ms)', 'chance, % of EODs within 10 ms')]:
    for j, (p, (c, mk)) in enumerate(PIPES.items()):
        t = RV[RV.pipeline == p].set_index('recording').loc[recs]
        xs = x + (j - 1.5) * (w + 0.01)
        ax.bar(xs, t[col], w, color=c, edgecolor='white', lw=1, label=p)
        ax.scatter(xs, t[ch], marker='_', s=260, color='0.15', lw=2, zorder=3, label='chance (labels shuffled, shares kept)' if j == 0 else None)
        for xx, v in zip(xs, t[col]): ax.text(xx, v, f'{v:.1f}', ha='center', va='bottom', fontsize=8, color='0.2')
    ax.set_xticks(x); ax.set_xticklabels([r.replace('T', ' ') + f" ({RECS[r]['tracking']})" for r in recs], fontsize=9)
    ax.set_ylabel(col); ax.set_axisbelow(True); ax.grid(axis='x', visible=False)
axs[0].set_title('refractory violations: EOD < 10 ms after the previous EOD, both assigned to the same fish', pad=34)
axs[0].legend(ncol=5, fontsize=9, loc='lower center', bbox_to_anchor=(0.5, 1.0), frameon=False)
plt.tight_layout(); fig.savefig(f'{OUT}/refractory_violations_4pipelines.png', dpi=120); plt.show()

/var/folders/w4/p225s7cd19n0j6tss7pdl21w0000gn/T/ipykernel_32719/2327313617.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); fig.savefig(f'{OUT}/refractory_violations_4pipelines.png', dpi=120); plt.show()
